# 02b – Generazione di `flows_prepared.parquet`

## Scopo
Questo notebook rigenera **solo** il file `flows_prepared.parquet`, il dataset preparato su cui lavorano i notebook di modellazione (`03_baseline`, `04_tokenization`).
Contiene esclusivamente i passaggi necessari a produrre il file, senza le analisi esplorative.

Le motivazioni di ogni scelta (soglia N, prefissi IP, porte di servizio, regola di orientamento, siti mancanti) e le evidenze numeriche che le giustificano sono in **`Preparation.ipynb`** (notebook 02) e nel riepilogo del progetto. Qui le trasformazioni sono riprodotte identiche, senza ripeterne la giustificazione.

## Quando eseguirlo
- Su una macchina dove `flows_prepared.parquet` non esiste ancora.
- Dopo una modifica alle regole di preparazione (in quel caso aggiornare anche `Preparation.ipynb`).

Basta eseguirlo **una volta**: i notebook successivi leggono direttamente il Parquet.

## Input e output
| | File | Note |
|---|---|---|
| Input | `data/student_dataset/flows.parquet` | 67.244.519 flow grezzi, 23 colonne |
| Input | `data/student_dataset/splits.json` | confini dello split cronologico 70/15/15 |
| Output | `data/student_dataset/flows_prepared.parquet` | 66.886.292 righe, 41 colonne, nessun NULL, ZSTD (~5,3 GB) |
| Output | `data/student_dataset/preparazione_config.json` | parametri stimati dal train (porte di servizio, intervalli, prefissi) |

Requisiti: kernel Python con `duckdb` e `pandas`; circa 5,5 GB liberi su disco. Tempo totale: qualche minuto.

## Contenuto del file prodotto
- `flow_id`, `t_start`, `split`; target `delta_rtt`; input numerico `start_rtt`.
- Colonne `an_*` (`an_end_rtt`, `an_endpoints_reversed`, `an_durata_s`): **solo per analisi, mai come input** dei modelli, perché dipendono dall'evento di fine del flow.
- Categoriche leggibili: `ip_version`, `experiment`, `exp_act`, `application`, `hostname`, siti e ASN CRIC (`MISSING` se mancanti), prefissi IP L1/L2/L3, porte, `porta_servizio`, `porta_client`, `verso` (`C2S`/`S2C`), `verso_incerto`, colonne canoniche `server_*` / `client_*`.
- **Nessuna soglia RARE/UNK applicata**: i vocabolari si costruiscono sul train nello stadio successivo (`vocab.py`).

## Nota sul leakage
L'unico parametro stimato dai dati è l'elenco delle porte di servizio, calcolato **solo sul train**. Pulizia, split e prefissi IP sono trasformazioni deterministiche e non usano informazioni di validation o test.

## 0. Impostazione
Percorsi relativi alla radice del progetto (`flow-tokenization`), supponendo che il notebook stia in una sua sottocartella (es. `results/` o `notebooks/`).

In [1]:
import json
import time
import ipaddress
from pathlib import Path

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd().parent                      # .../flow-tokenization
DATA_DIR = PROJECT_ROOT / "data" / "student_dataset"
FLOWS    = DATA_DIR / "flows.parquet"
SPLITS   = DATA_DIR / "splits.json"
OUT      = DATA_DIR / "flows_prepared.parquet"
CONFIG   = DATA_DIR / "preparazione_config.json"

SOVRASCRIVI = False      # True per rigenerare il file anche se esiste già

print("flows.parquet:", FLOWS.exists(), "| splits.json:", SPLITS.exists(), "| output già presente:", OUT.exists())

con = duckdb.connect()
con.execute("SET threads TO 8")
con.execute("SET memory_limit = '8GB'")
con.execute(f"SET temp_directory = '{PROJECT_ROOT / 'duckdb_tmp'}'")
con.execute("SET preserve_insertion_order = false")
con.execute("SET TimeZone = 'UTC'")
print("DuckDB", duckdb.__version__)

flows.parquet: True | splits.json: True | output già presente: False
DuckDB 2.0.0


## 1. Pulizia e split
- Si rimuovono i flow con `start_rtt = 0` o `end_rtt = 0` (RTT = 0 indica una misura mancante).
- Lo split è cronologico su `flow_start`, con i confini di `splits.json`.

**Valori attesi:** test 10.086.352, train 46.730.686, val 10.069.254.

In [2]:
con.execute(f"CREATE OR REPLACE VIEW flows AS SELECT * FROM read_parquet('{FLOWS}')")

with open(SPLITS) as f:
    splits = json.load(f)
TRAIN_END = splits["train_end_inclusive"]
VAL_END   = splits["validation_end_inclusive"]

con.execute(f"""
    CREATE OR REPLACE VIEW flows_clean AS
    SELECT
        *,
        CAST(flow_start AS TIMESTAMPTZ) AS t_start,
        CASE
            WHEN CAST(flow_start AS TIMESTAMPTZ) <= TIMESTAMPTZ '{TRAIN_END}' THEN 'train'
            WHEN CAST(flow_start AS TIMESTAMPTZ) <= TIMESTAMPTZ '{VAL_END}'   THEN 'val'
            ELSE 'test'
        END AS split
    FROM flows
    WHERE start_rtt > 0 AND end_rtt > 0
""")

con.execute("SELECT split, COUNT(*) AS n FROM flows_clean GROUP BY split ORDER BY split").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,split,n
0,test,10086352
1,train,46730686
2,val,10069254


## 2. Tabella IP → prefissi
Prefissi calcolati con `ipaddress` sugli IP unici (sorgente ∪ destinazione, tutti gli split). Gli IPv6 che incapsulano un IPv4 (`::ffff:a.b.c.d`) sono convertiti in IPv4.

| Livello | IPv4 | IPv6 |
|---|---|---|
| L1 | /16 | /32 |
| L2 (default) | /24 | /48 |
| L3 | /24 | /64 |

**Valori attesi:** 28.643 IP distinti, 0 non validi.

In [3]:
ips = con.execute("""
    SELECT "meta.src_ip" AS ip FROM flows_clean
    UNION
    SELECT "meta.dst_ip" FROM flows_clean
""").df()["ip"]

LIVELLI = {"L1": (16, 32), "L2": (24, 48), "L3": (24, 64)}   # (lunghezza IPv4, lunghezza IPv6)

def prefissi(s):
    """Dato un IP come stringa, restituisce versione e prefisso a ogni livello."""
    try:
        ip = ipaddress.ip_address(s)
    except ValueError:
        return {"version": None, **{k: None for k in LIVELLI}}
    if ip.version == 6 and ip.ipv4_mapped is not None:
        ip = ip.ipv4_mapped
    out = {"version": ip.version}
    for k, (l4, l6) in LIVELLI.items():
        L = l4 if ip.version == 4 else l6
        out[k] = str(ipaddress.ip_network(f"{ip}/{L}", strict=False))
    return out

ip_map = pd.DataFrame([prefissi(s) for s in ips])
ip_map.insert(0, "ip", ips.values)
con.register("ip_map", ip_map)

print("IP distinti:", len(ip_map), "| non validi:", ip_map["version"].isna().sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

IP distinti: 28643 | non validi: 0


## 3. Porte di servizio e macro SQL
- **Porte candidate:** porte < 32768 con almeno N = 1.000 flow nel **train**, in qualunque ruolo (sorgente o destinazione).
- **Porte nominate:** candidate fuori dagli intervalli 20000–24999 e 30000–30999; ogni intervallo diventa una categoria unica.
- Macro: `servizio(x)` (vero se `x` è una porta di servizio), `porta_cat(p)` (codifica della porta di servizio), `porta_fascia(p)` (fascia della porta client). L'intervallo effimero è quello Linux, 32768–60999.

**Valori attesi:** 537 porte candidate, 42 nominate.

In [4]:
cand = con.execute("""
    SELECT porta, SUM(n) AS n
    FROM (
        SELECT "meta.dst_port" AS porta, COUNT(*) AS n
        FROM flows_clean WHERE split = 'train' GROUP BY 1
        UNION ALL
        SELECT "meta.src_port", COUNT(*)
        FROM flows_clean WHERE split = 'train' GROUP BY 1
    )
    WHERE porta < 32768
    GROUP BY porta
    HAVING SUM(n) >= 1000
    ORDER BY n DESC
""").df()

in_range = cand["porta"].between(20000, 24999) | cand["porta"].between(30000, 30999)
NOMINATE = cand.loc[~in_range, "porta"].astype(int).tolist()
lista_nom = ", ".join(map(str, NOMINATE))
print("Porte candidate:", len(cand), "| nominate:", len(NOMINATE))

con.execute(f"""
    CREATE OR REPLACE MACRO servizio(x) AS
        x IN ({lista_nom})
        OR x BETWEEN 20000 AND 24999
        OR x BETWEEN 30000 AND 30999
""")

con.execute(f"""
    CREATE OR REPLACE MACRO porta_cat(p) AS
        CASE
            WHEN p BETWEEN 20000 AND 24999 THEN 'RANGE_20000'
            WHEN p BETWEEN 30000 AND 30999 THEN 'RANGE_30000'
            WHEN p IN ({lista_nom})         THEN CAST(p AS VARCHAR)
            WHEN p < 1024                   THEN 'WELL_KNOWN'
            WHEN p < 32768                  THEN 'REGISTERED'
            WHEN p <= 60999                 THEN 'EPHEMERAL'
            ELSE                                 'HIGH'
        END
""")

con.execute("""
    CREATE OR REPLACE MACRO porta_fascia(p) AS
        CASE
            WHEN p < 1024   THEN 'WELL_KNOWN'
            WHEN p < 32768  THEN 'REGISTERED'
            WHEN p <= 60999 THEN 'EPHEMERAL'
            ELSE                 'HIGH'
        END
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Porte candidate: 537 | nominate: 42


## 4. Costruzione e scrittura del file
Regola di orientamento a cascata:
1. porta di servizio in destinazione → `C2S` (client → server);
2. porta di servizio in sorgente → `S2C`;
3. una sola porta < 32768 → quella è il servizio;
4. altrimenti la porta più bassa, con `verso_incerto = TRUE`.

Le colonne `server_*` / `client_*` riordinano sito, ASN e prefissi secondo l'orientamento.
Il file non viene riscritto se esiste già, a meno che `SOVRASCRIVI = True`.

In [5]:
canoniche = []
for col in ["site", "asn", "L1", "L2", "L3"]:
    canoniche.append(f"CASE WHEN verso = 'C2S' THEN dst_{col} ELSE src_{col} END AS server_{col}")
    canoniche.append(f"CASE WHEN verso = 'C2S' THEN src_{col} ELSE dst_{col} END AS client_{col}")
canoniche_sql = ",\n        ".join(canoniche)

con.execute(f"""
    CREATE OR REPLACE VIEW flows_prepared AS
    WITH base AS (
        SELECT
            f.*,
            ms.L1 AS src_L1, ms.L2 AS src_L2, ms.L3 AS src_L3,
            md.L1 AS dst_L1, md.L2 AS dst_L2, md.L3 AS dst_L3,
            CASE
                WHEN servizio(f."meta.dst_port") THEN 'C2S'
                WHEN servizio(f."meta.src_port") THEN 'S2C'
                WHEN f."meta.dst_port" < 32768 AND f."meta.src_port" >= 32768 THEN 'C2S'
                WHEN f."meta.src_port" < 32768 AND f."meta.dst_port" >= 32768 THEN 'S2C'
                WHEN f."meta.dst_port" <= f."meta.src_port" THEN 'C2S'
                ELSE 'S2C'
            END AS verso,
            NOT (servizio(f."meta.dst_port")
                 OR servizio(f."meta.src_port")
                 OR (f."meta.dst_port" < 32768) <> (f."meta.src_port" < 32768)) AS verso_incerto
        FROM flows_clean AS f
        LEFT JOIN ip_map AS ms ON f."meta.src_ip" = ms.ip
        LEFT JOIN ip_map AS md ON f."meta.dst_ip" = md.ip
    ),
    pulito AS (
        SELECT
            flow_id, t_start, split,
            delta_rtt, start_rtt,
            end_rtt            AS an_end_rtt,
            endpoints_reversed AS an_endpoints_reversed,
            epoch(CAST(flow_end AS TIMESTAMPTZ)) - epoch(t_start) AS an_durata_s,
            "meta.ip_version"      AS ip_version,
            "meta.experiment_name" AS experiment,
            "meta.experiment_name" || '_' || CAST("meta.activity_id" AS VARCHAR) AS exp_act,
            COALESCE("meta.application", 'MISSING') AS application,
            "meta.syslog.hostname" AS hostname,
            COALESCE("meta.cric.src.name", 'MISSING') AS src_site,
            COALESCE("meta.cric.dst.name", 'MISSING') AS dst_site,
            CASE WHEN "meta.cric.src.asn" IS NULL OR "meta.cric.src.asn" = 0
                 THEN 'MISSING' ELSE CAST("meta.cric.src.asn" AS VARCHAR) END AS src_asn,
            CASE WHEN "meta.cric.dst.asn" IS NULL OR "meta.cric.dst.asn" = 0
                 THEN 'MISSING' ELSE CAST("meta.cric.dst.asn" AS VARCHAR) END AS dst_asn,
            src_L1, src_L2, src_L3, dst_L1, dst_L2, dst_L3,
            "meta.src_port" AS src_port,
            "meta.dst_port" AS dst_port,
            "meta.src_ip"   AS src_ip,
            "meta.dst_ip"   AS dst_ip,
            verso, verso_incerto,
            CASE WHEN verso = 'C2S' THEN "meta.dst_port" ELSE "meta.src_port" END AS porta_servizio_num,
            CASE WHEN verso = 'C2S' THEN "meta.src_port" ELSE "meta.dst_port" END AS porta_client_num
        FROM base
    )
    SELECT
        * EXCLUDE (porta_servizio_num, porta_client_num),
        porta_cat(porta_servizio_num)  AS porta_servizio,
        porta_fascia(porta_client_num) AS porta_client,
        {canoniche_sql}
    FROM pulito
""")

if OUT.exists() and not SOVRASCRIVI:
    print(f"{OUT.name} esiste già: scrittura saltata (SOVRASCRIVI = False).")
else:
    t0 = time.time()
    con.execute(f"COPY (SELECT * FROM flows_prepared) TO '{OUT}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    print(f"Scritto {OUT.name} in {time.time() - t0:.0f} s")

    config_prep = {
        "N_soglia": 1000,
        "porte_servizio_nominate": NOMINATE,
        "intervalli_porte_servizio": [[20000, 24999], [30000, 30999]],
        "intervallo_effimero_linux": [32768, 60999],
        "prefissi": {"L1": [16, 32], "L2": [24, 48], "L3": [24, 64]},
    }
    with open(CONFIG, "w") as fh:
        json.dump(config_prep, fh, indent=2)
    print(f"Scritto {CONFIG.name}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Scritto flows_prepared.parquet in 113 s
Scritto preparazione_config.json


## 5. Verifica
**Valori attesi** (verificati il 14/11/2026):
- righe per split: test 10.086.352, train 46.730.686, val 10.069.254;
- `errori_canoniche` 0, `pct_S2C` 17,773, `pct_incerto` 0,266, `durate_negative` 0 (sul train);
- 41 colonne, circa 5,3 GB.

In [6]:
con.execute(f"CREATE OR REPLACE VIEW prep AS SELECT * FROM read_parquet('{OUT}')")

display(con.execute("SELECT split, COUNT(*) AS n FROM prep GROUP BY split ORDER BY split").df())

display(con.execute("""
    SELECT
        SUM(CASE WHEN verso = 'C2S' AND (server_L2 <> dst_L2 OR client_L2 <> src_L2) THEN 1 ELSE 0 END)
      + SUM(CASE WHEN verso = 'S2C' AND (server_L2 <> src_L2 OR client_L2 <> dst_L2) THEN 1 ELSE 0 END)
            AS errori_canoniche,
        ROUND(100 * AVG(CASE WHEN verso = 'S2C' THEN 1 ELSE 0 END), 3) AS pct_S2C,
        ROUND(100 * AVG(CAST(verso_incerto AS INTEGER)), 3)            AS pct_incerto,
        SUM(CASE WHEN an_durata_s < 0 THEN 1 ELSE 0 END)               AS durate_negative
    FROM prep
    WHERE split = 'train'
""").df())

n_col = len(con.execute("DESCRIBE prep").df())
print("Colonne:", n_col, "| dimensione:", f"{OUT.stat().st_size / 1e9:.2f} GB")

,split,n
0,test,10086352
1,train,46730686
2,val,10069254


,errori_canoniche,pct_S2C,pct_incerto,durate_negative
0,0.0,17.773,0.266,0.0


Colonne: 41 | dimensione: 5.29 GB


In [7]:
%pip install lightgbm scikit-learn scipy torch

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ----------------------- ---------------- 0.8/1.4 MB 6.0 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 5.9 MB/s  0:00:00
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ------ --------------------------------- 1.3/8.4 MB 8.9 MB/s eta 0:00:01
   --------------- ------------------------ 3.1/8.4 MB 8.5 MB/s eta 0:00:01
   ------------------------- -------------- 5.2/8.4 MB 9.2 MB/s eta 0:00:01
   --------------------------------- ------ 7.1/8.4 MB 9.4 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 8.5 MB/s  0:00:01
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 1.6/126.7 MB 8.5 MB/s eta 0:00:15
   - -------------------------------------- 3.4/126.7 MB 8.3 MB/s eta 0:00:15
   - -------------------------------------- 5.0/126.7 MB 8.0 MB/s eta 0:00:16
   -- -------------------------


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import lightgbm as lgb
import sklearn
import scipy
import torch

print("lightgbm", lgb.__version__, "| scikit-learn", sklearn.__version__,
      "| scipy", scipy.__version__, "| torch", torch.__version__)